# Compute SUMMA soil parameters 
This is untracked
**Please clean all outputs when committing all the changes in git.**

In [ ]:
%matplotlib inline  
import os, sys
import xarray as xr
import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from sklearn import linear_model

import scripts.soil_attributes as soil
import scripts.utility as util

print("\nThe Python version: %s.%s.%s" % sys.version_info[:3])

-------------------------
## 1. Setup <a id='setup'></a>

In [ ]:
setup = util.load_yaml("./summa_soil_param_soilgrid.yaml")

catchment_name = setup["catchment_name"]  # Catchment case: CONUS_HUC12, camels, camelsx-v3
serial = setup["serial"] # True: dask distributed enabled
saveCSV = setup["saveCSV"] # True: save attributes for each HRU in csv
saveNetCDF = setup["saveNetCDF"] # True: save attributes for each HRU in netcdf
remap = setup["remap"] # True: remap meteorological time series to HRUs

# files and directories
src_sg_dir = setup["src_sg_dir"]
catch_gpkg = setup["catch_gpkg"]
sg_mapping_file = setup["sg_mapping_file"]

# soilgrid meta
nc_sg_dict = setup['nc_sg_dict']
soilgrid_thickness = setup['soilgrid_thickness']
target_thickness = setup['target_thickness']

# catchment meta
catch_attrs = setup["catch_attrs"]

print('-- Setup:')
print(f" Dask not enabled: {serial}")
print(f" catchment_name: {catchment_name}")
print(f" saveCSV: {saveCSV}")
print(f" saveNetCDF: {saveNetCDF}")
print(f" remap: {remap}")
print(f" soilgrid data directory: {src_sg_dir}")
print(f" catchment gpkg: {catch_gpkg}")
print(f" sg_mapping_file: {sg_mapping_file}")

## Reading hru geopackage

In [ ]:
gdf_catch = gpd.read_file(catch_gpkg)
gdf_catch = gdf_catch.rename(columns={catch_attrs[catchment_name]['id']:'hruId'})
gdf_catch = gdf_catch.set_index('hruId')

# Reading soilgrids data

In [ ]:
%%time
print(f'Reading soilgrid data')
for ix, (header, variable) in enumerate(nc_sg_dict.items()):
    print(f'{ix} {variable} in {header}')
    if ix==0:
        ds = xr.open_dataset(os.path.join(src_sg_dir, f'{header}_LOCA2.nc')).load()
    else:
        ds = ds.merge(xr.open_dataset(os.path.join(src_sg_dir, f'{header}_LOCA2.nc')).load())
ds = ds.drop_vars(['crs'])
#ds = ds.apply(lambda da: da.transpose('lat', 'lon', 'lyr') if set(['lyr', 'lat', 'lon']).issubset(da.dims) else da)

ds['bedrock_depth'].values = ds['bedrock_depth'].values*0.01 # cm -> m
ds['bulk_density'].values = ds['bulk_density'].values*0.01 # cg/cm3 -> g/cm3
ds['silt_pct'].values = ds['silt_pct'].values*0.1 # g/kg -> percent
ds['clay_pct'].values = ds['clay_pct'].values*0.1 # g/kg -> percent
ds['sand_pct'].values = ds['sand_pct'].values*0.1 # g/kg -> percent
ds['occ'].values = ds['occ'].values*0.01 # dg/kg -> percent

ds['bedrock_depth'].attrs['units'] = 'm'
ds['occ'].attrs['units'] = 'percent'
ds['silt_pct'].attrs['units'] = 'percent'
ds['clay_pct'].attrs['units'] = 'percent'
ds['sand_pct'].attrs['units'] = 'percent'
ds['bulk_density'].attrs['units'] = 'g/cm3'

In [ ]:
%%time
ds['usda_soil_class'] = soil.USDA_soil_classification(ds['sand_pct'], ds['clay_pct'], ds['silt_pct'])

In [ ]:
%%time
ds['sat_matric_potential'] = soil.matric_potential_ptf1(ds['sand_pct'], ds['silt_pct'])
ds['retention_slope'] = soil.retention_slope_ptf1(ds['sand_pct'], ds['clay_pct'])

ds['theta_sat'] = soil.porosity_ptf1(ds['usda_soil_class'])
#ds['theta_sat'] = soil.porosity_ptf3(ds['sand_pct'], ds['clay_pct'])

ds['k_soil'] = soil.k_sat_ptf1(ds['sand_pct'], ds['clay_pct'])

ds['fieldCapacity'] = soil.field_capacity_ptf1(ds['usda_soil_class'])
#ds['fieldCapacity'] = soil.field_capacity_ptf3(ds['theta_sat'], ds['sat_matric_potential'],ds['retention_slope'])

ds['critSoilWilting'] = soil.wilting_point_ptf1(ds['usda_soil_class'])
#ds['critSoilWilting'] = soil.wilting_point_ptf3(ds['theta_sat'], ds['sat_matric_potential'],ds['retention_slope'])

ds['vGn_alpha'] = soil.vGn_alpha_ptf1(ds['sand_pct'], ds['clay_pct'], ds['bulk_density'], ds['occ'])
ds['vGn_n'] = soil.vGn_n_ptf1(ds['sand_pct'], ds['clay_pct'])
#ds['wettingFrontSuction'] = soil.wettingFrontSuction_tf1(ds['usda_soil_class'])

## Re-mapping

In [ ]:
%%time
dr_mask = xr.where(np.isnan(ds['silt_pct'].isel(lyr=0)), 0, 1)
soil_vars = ['theta_sat', 'k_soil','fieldCapacity', 'critSoilWilting', 'vGn_alpha', 'vGn_n']
a1 = []
a2 = []
for ly in np.arange(6):
    a1.append(util.regrid_mean(xr.open_dataset(sg_mapping_file), ds.isel(lyr=ly), dr_mask, soil_vars, verbose=False))
    print('finished %d'%ly)
a = xr.concat(a1, dim="lyr")
a = xr.merge([a, util.regrid_mean(xr.open_dataset(sg_mapping_file), ds, xr.where(np.isnan(ds['bedrock_depth']), 0, 1), ['bedrock_depth'], verbose=False)]) # remap bedrock to depth [m] data

In [ ]:
# vertical, layer aggregation
# take a mean over layers for soil_vars
b = a.mean(dim='lyr')
#b=xr.Dataset()
#for var in soil_vars:
#    b = xr.merge([b, util.remap_mean_vertical(layer_mapping, a[var], default=np.nan)])

## fill missing values hru

In [ ]:
# find hru with missing value
missing_huc = b['hru'].where(b['k_soil']==0, drop=True).values

In [ ]:
gdf_catch_proj = gdf_catch.to_crs('epsg:3785')
centroid = gdf_catch_proj.copy(deep=True)
centroid.geometry = centroid['geometry'].centroid

print('index missing_huc12, target_closest_huc12')
count=0
neighbor_huc12 = {}
for index, row in centroid.iterrows():
    if index not in missing_huc:
        continue
    count+=1
    geo_ix = gdf_catch_proj.geometry.distance(row.geometry).sort_values().index[1:50]
    for ix in geo_ix:
        if ix not in missing_huc:
            neighbor_huc12[index] = ix
            break
    if count%10==0:
        print(count, index, neighbor_huc12[index])

In [ ]:
soil_vars = ['theta_sat', 'k_soil','fieldCapacity', 'critSoilWilting', 'vGn_alpha', 'vGn_n']
for huc12_missing, huc12_target in neighbor_huc12.items():
    for var in soil_vars:
        b[var].loc[dict(hru=huc12_missing)] = b[var].loc[dict(hru=huc12_target)].values

## Compute critSoilTranspire and theta_res

In [ ]:
b['theta_res'] = 0.1*b['critSoilWilting']
b['critSoilTranspire'] = 0.02+ b['critSoilWilting']

## Enforce parameter values within range

In [ ]:
def read_localParam(ascii):
    parameter = {}
    with open(ascii, "r") as f:
        for line in f:
            line = line.strip()
            # skip empty or comment lines
            if not line or line.startswith("!"):
                continue
            if 'format string' in line:
                continue
                
            # split on '|'
            parts = [p.strip() for p in line.split("|")]
            
            # sanity check: we expect 4 parts (name + 3 numbers)
            if len(parts) != 4:
                continue
            
            name, default, min_val, max_val = parts
            default = default.replace("d", "e")
            min_val = min_val.replace("d", "e")
            max_val = max_val.replace("d", "e")
            parameter[name] = {
                "default": float(default),
                "min": float(min_val),
                "max": float(max_val)
            }
    return parameter

In [ ]:
localParamInfo = '/glade/u/home/mizukami/proj/cmip6_hydro/models/prep_summa_settings/summa_settings/localParamInfo.txt'
localParam = read_localParam(localParamInfo)
localParam['critSoilWilting']['min'] = 0.01 # currently min is 0, which is strange

for par in ['theta_sat', 'k_soil','fieldCapacity', 'critSoilWilting', 'vGn_alpha', 'vGn_n', 'critSoilTranspire', 'theta_res']:
    b[par] = b[par].where(b[par]>=localParam[par]['min'], localParam[par]['min'])
    b[par] = b[par].where(b[par]<=localParam[par]['max'], localParam[par]['max'])

#### Final check SUMMA soil parameter consistencies

In [ ]:
(b['theta_res']>=b['theta_sat']).values.any()

In [ ]:
(b['critSoilTranspire']<=b['critSoilWilting']).values.any()

In [ ]:
(b['theta_sat']<=b['critSoilWilting']).values.any()

In [ ]:
(b['theta_res']>=b['critSoilWilting']).values.any()

In [ ]:
(b['theta_sat']<=b['fieldCapacity']).values.any()

In [ ]:
(b['theta_res']>=b['fieldCapacity']).values.any()

In [ ]:
(b['theta_sat']<=b['critSoilTranspire']).values.any()

In [ ]:
(b['theta_res']>=b['critSoilTranspire']).values.any()

In [ ]:
for par in ['theta_sat', 'k_soil','fieldCapacity', 'critSoilWilting', 'vGn_alpha', 'vGn_n', 'theta_res', 'critSoilTranspire']:
    if (b[par]>=localParam[par]['min']).values.all() or (b[par] <= localParam[par]['max']).values.all():
        print(f'{par} looks good')

## Dataset to Dataframe

In [ ]:
df = b.to_dataframe()

## Save in csv or netcdf

In [ ]:
if saveCSV :
    df.to_csv(os.path.join('output',f'{catchment_name}_summa_soil_parameter1.csv'), float_format='%g')
if saveNetCDF:
    b.to_netcdf(os.path.join('output',f'{catchment_name}_summa_soil_parameter1.nc'))

## Plotting...

In [ ]:
gdf_catch = gdf_catch.merge(df,left_index=True, right_index=True)

In [ ]:
var_name = 'theta_sat'
fig, ax = plt.subplots(figsize=(6, 3), dpi=150)
#gdf_camels.plot(ax=ax, color='white', edgecolor='black', lw=0.3)
gdf_catch.plot(ax=ax, column=var_name, cmap='turbo',
                norm=colors.Normalize(0.3600, 0.5100),
                legend=True
               );
ax.set_title(var_name);
fig.tight_layout()
plt.savefig(f'./figures/{catchment_name}_summa_soil_{var_name}_pft3.png', dpi=300)

In [ ]:
var_name = 'fieldCapacity'
fig, ax = plt.subplots(figsize=(6, 3), dpi=150)
gdf_catch.plot(ax=ax, column=var_name, cmap='turbo',
                norm=colors.Normalize(0.28, 0.42),
                legend=True
               );
ax.set_title(var_name);
fig.tight_layout()
plt.savefig(f'./figures/{catchment_name}_summa_soil_{var_name}_pft3.png', dpi=300)

In [ ]:
var_name = 'critSoilWilting'
fig, ax = plt.subplots(figsize=(6, 3), dpi=150)
#gdf_catch.plot(ax=ax, color='white', edgecolor='black', lw=0.3)
gdf_catch.plot(ax=ax, column=var_name, cmap='turbo',
                norm=colors.Normalize(0.05, 0.28),
                legend=True
               );
ax.set_title(var_name);
fig.tight_layout()
plt.savefig(f'./figures/{catchment_name}_summa_soil_{var_name}_pft3.png', dpi=300)

In [ ]:
var_name = 'critSoilTranspire'
fig, ax = plt.subplots(figsize=(6, 3), dpi=150)
#gdf_catch.plot(ax=ax, color='white', edgecolor='black', lw=0.3)
gdf_catch.plot(ax=ax, column=var_name, cmap='turbo',
                norm=colors.Normalize(0.05, 0.30),
                legend=True
               );
ax.set_title(var_name);
fig.tight_layout()
plt.savefig(f'./figures/{catchment_name}_summa_soil_{var_name}_pft3.png', dpi=300)

In [ ]:
var_name = 'k_soil'
fig, ax = plt.subplots(figsize=(6, 3), dpi=150)
gdf_catch.plot(ax=ax, column=var_name, cmap='turbo',
                norm=colors.Normalize(1.e-7, 1.e-5),
                legend=True
               );
ax.set_title(var_name);
fig.tight_layout()
plt.savefig(f'./figures/{catchment_name}_summa_soil_{var_name}.png', dpi=300)

In [ ]:
var_name = 'vGn_alpha'
fig, ax = plt.subplots(figsize=(6, 3), dpi=150)
gdf_catch.plot(ax=ax, column=var_name, cmap='turbo',
                norm=colors.Normalize(-1.00, -0.01),
                legend=True
               );
ax.set_title(var_name);
fig.tight_layout()
plt.savefig(f'./figures/{catchment_name}_summa_soil_{var_name}.png', dpi=300)

In [ ]:
var_name = 'vGn_n'
fig, ax = plt.subplots(figsize=(6, 3), dpi=150)
gdf_catch.plot(ax=ax, column=var_name, cmap='turbo',
                norm=colors.Normalize(1, 3),
                legend=True
               );
ax.set_title(var_name);
fig.tight_layout()
plt.savefig(f'./figures/{catchment_name}_summa_soil_{var_name}.png', dpi=300)

In [ ]:
var_name = 'theta_res'
fig, ax = plt.subplots(figsize=(6, 3), dpi=150)
gdf_catch.plot(ax=ax, column=var_name, cmap='turbo',
                norm=colors.Normalize(0.01, 0.05),
                legend=True
               );
ax.set_title(var_name);
fig.tight_layout()
plt.savefig(f'./figures/{catchment_name}_summa_soil_{var_name}.png', dpi=300)